# 02 — Exploratoire : forces, bêtes noires, pronostics ELO, nationalité

Quatre questions, sur les **deux saisons complètes** (23-24, 24-25) pour les résultats,
sur les **11 saisons** pour le panorama joueuses.

- **A.** Quelles équipes sont armées pour gagner ?
- **B.** Existe-t-il des « bêtes noires » (confrontations anormales) ?
- **C.** Peut-on pronostiquer une saison avec l'ELO ?
- **D.** Les joueuses étrangères changent-elles le jeu ? + panorama (COVID, âge).

In [1]:
import sys
from pathlib import Path
p = Path.cwd()
while not (p / "analyse" / "outils.py").exists() and p != p.parent:
    p = p.parent
sys.path.insert(0, str(p / "analyse"))
import numpy as np, pandas as pd, outils as o
OUT = o.RESULTATS / "exploratoire"; OUT.mkdir(parents=True, exist_ok=True)
pd.set_option("display.width", 140); pd.set_option("display.max_columns", 30)
print("sorties ->", OUT)

sorties -> C:\Users\TEMP.IUT-LUMIERE.000\R6.06-Domaines-d-application-de-la-statistique\analyse\resultats\exploratoire


In [2]:
df = o.charger_equipes(o.SAISONS_COMPLETES)
t = o.agreger_equipe_match(df)
eps = 1e-9
t["eFG"]         = (t["Tirs_marques"] + 0.5*t["3pts_marques"]) / (t["Tirs_tentes"]+eps)
t["pct_3pts"]    = t["3pts_marques"] / (t["3pts_tentes"]+eps)
t["poss"]        = t["Tirs_tentes"] - t["RO"] + t["BP"] + 0.44*t["LF_tentes"]
t["taux_pertes"] = t["BP"] / (t["poss"]+eps)
t["pression_def"]= t["INT"] + t["CT"]
m = o.differentiel_adversaire(t, ["eFG","pct_3pts","taux_pertes","RD","RO","PD","pression_def","Pts"])
print(len(m), "confrontations")

358 confrontations


## A. Quelles équipes sont armées pour gagner ?

Par équipe et saison, l'**avantage moyen sur l'adversaire** (« net ») sur chaque
facteur, et sa corrélation avec le taux de victoire.

In [3]:
g = (m.groupby(["eq","Saison"]).agg(
        matchs=("Victoire","size"), taux_victoire=("Victoire","mean"),
        net_points=("d_Pts","mean"), net_eFG=("d_eFG","mean"),
        net_3pts=("d_pct_3pts","mean"), net_RD=("d_RD","mean"),
        net_PD=("d_PD","mean"), net_pertes=("d_taux_pertes","mean"))
     .reset_index())
g = g[g["matchs"] >= 5]
g.to_csv(OUT / "forces_equipe_saison.csv", index=False)
corr = (g[["net_points","net_eFG","net_3pts","net_RD","net_PD","net_pertes"]]
        .corrwith(g["taux_victoire"]).sort_values(ascending=False).round(2))
print("Correlation avantage-facteur <-> %% victoires (n=%d eq-saisons) :" % len(g))
print(corr.to_string())
print()
print(g.sort_values(["Saison","net_points"], ascending=[True,False]).round(2).head(12).to_string(index=False))

Correlation avantage-facteur <-> % victoires (n=20 eq-saisons) :
net_points    0.94
net_eFG       0.85
net_PD        0.79
net_RD        0.68
net_3pts      0.62
net_pertes   -0.40

             eq Saison  matchs  taux_victoire  net_points  net_eFG  net_3pts  net_RD  net_PD  net_pertes
villeneuve ascq  23-24      18           0.89       17.11     0.11      0.09    5.67    3.72       -0.01
        bourges  23-24      18           0.78       11.00     0.06      0.01    3.22    5.28       -0.02
  basket landes  23-24      18           0.67        8.67     0.03      0.00    4.50    2.28       -0.02
           lyon  23-24      18           0.44       -0.28     0.03      0.01    1.67    1.67        0.04
        charnay  23-24      18           0.44       -2.39    -0.01     -0.05   -5.50   -4.00       -0.05
    montpellier  23-24      18           0.50       -3.11    -0.03     -0.02   -1.17   -1.56        0.00
         angers  23-24      18           0.33       -4.11    -0.09     -0.04   -4.67 

**Lecture.** L'efficacité au tir et les passes décisives sont les plus liées à la
réussite d'une équipe. **Bourges** domine (net points le plus élevé les deux saisons).

## B. Bêtes noires

Pour chaque paire, on compare le % de victoires réel au % **attendu** vu les niveaux
respectifs (Bradley-Terry simplifié). Sous-performance marquée = bête noire.

In [4]:
niveau = m.groupby("eq")["Victoire"].mean()
pair = (m.groupby(["eq","eq_opp"]).agg(n=("Victoire","size"), vic=("Victoire","mean"),
                                       net=("d_Pts","mean")).reset_index())
pair = pair[pair["n"] >= 3].copy()
pair["niv_eq"]  = pair["eq"].map(niveau); pair["niv_opp"] = pair["eq_opp"].map(niveau)
pair["attendu"] = pair["niv_eq"] / (pair["niv_eq"] + pair["niv_opp"] + 1e-9)
pair["sous_perf"] = pair["vic"] - pair["attendu"]
pair = pair.sort_values("sous_perf").reset_index(drop=True)
pair.to_csv(OUT / "confrontations.csv", index=False)
print("ATTENTION : 3-4 confrontations par paire -> tendance, pas significatif.\n")
print(pair.head(8)[["eq","eq_opp","n","vic","attendu","sous_perf","net"]].round(2).to_string(index=False))

ATTENTION : 3-4 confrontations par paire -> tendance, pas significatif.

             eq        eq_opp  n  vic  attendu  sous_perf    net
     landerneau  roche vendee  3 0.00     0.60      -0.60  -8.67
    montpellier       bourges  4 0.00     0.41      -0.41 -13.75
           lyon       bourges  4 0.00     0.37      -0.37 -10.75
         angers       bourges  4 0.00     0.35      -0.35 -10.00
   roche vendee        angers  4 0.00     0.34      -0.34 -14.25
     landerneau basket landes  4 0.00     0.33      -0.33 -20.25
   roche vendee   charleville  4 0.00     0.33      -0.33  -6.50
villeneuve ascq          lyon  4 0.25     0.55      -0.30   4.00


**Lecture.** **Bourges est la bête noire de presque toute la ligue** : plusieurs bonnes
équipes ne la battent jamais. À nuancer : 3-4 matchs par paire seulement.

## C. Pronostiquer la saison 24-25 avec l'ELO (backtest hors-échantillon)

On prédit les 132 matchs de championnat 24-25 avec de l'info connue **avant** chaque
match (favori ELO), et on compare au résultat réel.

In [5]:
cal = o.charger_calendrier(forme=False)
c24 = cal[cal["Saison"] == "24-25"]
acc_naif = c24["home_win"].mean()
pred_elo = (c24["d_elo"] > 0).astype(int)
acc_elo  = (pred_elo == c24["home_win"]).mean()
upset    = (pred_elo != c24["home_win"]).mean()
print(f"{len(c24)} matchs 24-25")
print(f"  'le domicile gagne' (naif) : {acc_naif*100:.1f}%")
print(f"  favori ELO (avant match)   : {acc_elo*100:.1f}%")
print(f"  -> upsets (le favori perd)  : {upset*100:.0f}% des matchs")

132 matchs 24-25
  'le domicile gagne' (naif) : 59.8%
  favori ELO (avant match)   : 68.2%
  -> upsets (le favori perd)  : 32% des matchs


**Lecture.** L'ELO atteint ~68 % de bons pronostics (vs ~58 % en naïf) : **oui on peut
pronostiquer**, mais ~1 match sur 3 est une surprise. Le modèle abouti est au notebook 05.

## D. Nationalité & panorama des joueuses (11 saisons)

In [6]:
L = o.lire_liste_joueuses()
ev = (L.groupby("Saison").agg(n=("cle","size"),
        pct_etr=("francaise", lambda s:(1-s.mean())*100),
        age=("age_num","mean"), taille=("taille_num","mean")).reset_index())
ev["ord"] = ev["Saison"].str[:2].astype(int)
ev = ev.sort_values("ord").drop(columns="ord")
ev.to_csv(OUT / "evolution_saison.csv", index=False)
print(ev.round(2).to_string(index=False))

Saison   n  pct_etr   age  taille
 15-16 188    33.51 23.89    1.81
 16-17 162    33.95 24.35    1.81
 17-18 177    34.46 24.43    1.81
 18-19 177    33.90 24.66    1.82
 19-20 164    35.37 24.70    1.82
 20-21 174    39.08 25.04    1.82
 21-22 181    33.15 24.92    1.81
 22-23 185    35.14 24.56    1.82
 23-24 180    36.67 24.62    1.81
 24-25 171    31.58 24.30    1.81
 25-26 168    29.17 23.86    1.80


**COVID** (19-20, 20-21) : pas d'effondrement des étrangères (léger pic même en 20-21),
puis repli récent. **Âge** stable ~24-25 ans → pas de rajeunissement net.

In [7]:
nat = L[["cle","Saison","francaise"]].drop_duplicates(["cle","Saison"])
j = df[df["Joueur"] != "TOTAUX EQUIPE"].copy()
for c in ["Tps_jeu_decimal","Pts","PD","RT","INT","BP","EVAL"]:
    j[c] = pd.to_numeric(j[c].astype(str).str.replace(",",".",regex=False), errors="coerce")
j = j.merge(nat, on=["cle","Saison"], how="left")
print(f"appariement nationalite : {j['francaise'].notna().mean()*100:.0f}% des lignes\n")
js = (j.dropna(subset=["francaise"]).groupby(["cle","Saison","francaise"])
        .agg(matchs=("Pts","size"), minutes=("Tps_jeu_decimal","mean"), pts=("Pts","mean"),
             passes=("PD","mean"), rebonds=("RT","mean"), eval=("EVAL","mean")).reset_index())
js = js[js["matchs"] >= 5]
prof = js.groupby("francaise").agg(n=("pts","size"), minutes=("minutes","mean"),
        points=("pts","mean"), passes=("passes","mean"), rebonds=("rebonds","mean"),
        eval=("eval","mean"))
prof.index = ["Etrangere","Francaise"]
prof.to_csv(OUT / "profil_individuel.csv")
print(prof.round(2).to_string())

appariement nationalite : 83% des lignes

             n  minutes  points  passes  rebonds   eval
Etrangere   87    25.23    9.32    2.00     4.35  10.12
Francaise  151    17.19    4.92    1.36     2.21   5.31


**Lecture.** Les étrangères jouent ~1,5× plus et ont un impact (EVAL) ~2× supérieur :
ce sont les **cadres** recrutées ; les françaises forment le réservoir (banc, rotation).

**Conclusion.** Bourges = patron ; l'ELO pronostique honnêtement (~68 %) ; les étrangères
sont les meilleures joueuses mais ne créent pas mécaniquement la victoire (les gros
budgets recrutent mieux ET gagnent). Résultats dans `resultats/exploratoire/`.